# Lab 11 — False-Negative Forensics & Error Slicing

*AI Cybersecurity Masterclass · Lab 11 · AI60secLab*

**Companion video:** none yet for this lab.

**Research question:** Which attacks does the model miss, and what patterns distinguish those misses from correctly detected attacks?

Aggregate metrics tell us how many errors occurred. Forensics studies where those errors concentrate.

## Security foundation: threat hunting

Detection waits for alerts. **Threat hunting** goes looking for attackers the alerts missed. Hunts are usually **hypothesis-driven**: "if attackers were exfiltrating data over DNS, we would see unusually long subdomains from a few hosts. Let's look."

After an incident, teams also run a **detection gap analysis**: which steps of the attack did our tools miss, and why? The answers become new detections.

False-negative forensics is exactly this: study what got through, find the pattern, and turn it into a hypothesis you can test.

In [1]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

Project root: /home/claude/run/exec
Data folder : /home/claude/run/exec/data/raw


In [2]:
from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.forensics import (
    attach_prediction_context,
    error_slice_by_category,
    numeric_false_negative_contrast,
)

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

model = random_forest_pipeline(X_train)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
scores = model.predict_proba(X_test)[:, 1]

forensic = attach_prediction_context(
    test,
    y_test,
    predictions,
    scores,
)

display(forensic["error_type"].value_counts().to_frame())

,count
error_type,
true_positive,44768
true_negative,26927
false_positive,10073
false_negative,564


In [3]:
category_report = error_slice_by_category(
    forensic,
    "attack_cat",
)

display(category_report.round(4))

,attack_count,detected_count,mean_attack_score,missed_count,recall
attack_cat,,,,,
Fuzzers,6062,5546,0.7798,516,0.9149
Shellcode,378,375,0.9411,3,0.9921
Exploits,11132,11091,0.9709,41,0.9963
DoS,4089,4086,0.9788,3,0.9993
Reconnaissance,3496,3495,0.9824,1,0.9997
Generic,18871,18871,0.9931,0,1.0000
Analysis,677,677,0.9532,0,1.0000
Backdoor,583,583,0.9646,0,1.0000
Worms,44,44,0.9651,0,1.0000


In [4]:
false_negatives = forensic[
    forensic["error_type"] == "false_negative"
].copy()

columns_to_show = [
    c for c in [
        "attack_cat",
        "proto",
        "service",
        "state",
        "attack_score",
    ]
    if c in false_negatives.columns
]

display(
    false_negatives[columns_to_show]
    .sort_values("attack_score")
    .head(25)
)

,attack_cat,proto,service,state,attack_score
2672,Fuzzers,udp,-,INT,0.004000
6208,Fuzzers,udp,-,INT,0.004000
6147,Fuzzers,udp,-,INT,0.004000
6213,Fuzzers,udp,-,INT,0.004000
6194,Fuzzers,udp,-,INT,0.012000
4944,Fuzzers,udp,-,INT,0.020000
4838,Fuzzers,udp,-,INT,0.025938
6458,Fuzzers,tcp,-,FIN,0.048000
4831,Fuzzers,udp,-,INT,0.056000
4835,Fuzzers,udp,-,INT,0.062037


In [5]:
candidate_numeric = [
    "dur",
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "rate",
    "sload",
    "dload",
    "sinpkt",
    "dinpkt",
]

contrast = numeric_false_negative_contrast(
    forensic,
    candidate_numeric,
)

display(contrast.round(4))

,feature,false_negative_median,true_positive_median,difference
6,sload,10836.7324,5.066666e+07,-5.065583e+07
5,rate,32.7234,1.000000e+05,-9.996728e+04
7,dload,3162.2711,0.000000e+00,3.162271e+03
1,sbytes,738.0000,2.000000e+02,5.380000e+02
2,dbytes,268.0000,0.000000e+00,2.680000e+02
9,dinpkt,68.4998,0.000000e+00,6.849980e+01
8,sinpkt,50.9799,9.000000e-03,5.097090e+01
3,spkts,10.0000,2.000000e+00,8.000000e+00
4,dpkts,6.0000,0.000000e+00,6.000000e+00
0,dur,0.5243,0.000000e+00,5.243000e-01


## Build a research hypothesis

Choose one weak attack category.

Write:
- **Observation**
- **Hypothesis**
- **Experiment to test the hypothesis**
- **Alternative explanation**
- **Dataset limitation**

## From lab to SOC

Here, more than 90% of misses came from one category (Fuzzers), and many looked like ordinary conversations. In a SOC that becomes a hunting hypothesis and a detection backlog item, not a footnote under an aggregate recall number.

**Check yourself**
1. Turn this lab's main finding into a hunt hypothesis an analyst could run.
2. Why are false negatives harder to study in production than in a lab?
3. Which data source would you add to catch the missed Fuzzers?